# Daily H=22, notebook 2/2: Optuna, rolling neural e avaliação

Este notebook recebe o ZIP de transferência do notebook 1, executa a busca de
hiperparâmetros e usa imediatamente a configuração vencedora para ajustar
TSMixerX e LSTM nas 16 janelas 8/2/1. Depois gera explicabilidade, métricas,
testes e o pacote final.

O preset final usa 25 trials, até 400 passos, pruning e early stopping.
**Tempo estimado no Colab com GPU:** 50–65 minutos. Nenhum arquivo é gravado no
Google Drive; o ZIP final é baixado pelo navegador.

In [ ]:
# Configuração compartilhada: repositório + armazenamento local da sessão
from pathlib import Path
import os, shutil, subprocess, sys

REPO_URL = 'https://github.com/hugogobato/Mestrado_Anna_Julia.git'
REPO = Path('/content/Mestrado_Anna_Julia')
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

EXP = REPO / 'experiments/daily_h22'
PERSIST = Path('/content/daily_h22_run')
DATA_DIR = PERSIST / 'data'
RESULTS_DIR = PERSIST / 'results'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
os.environ['DAILY_H22_DATA_DIR'] = str(DATA_DIR)
os.environ['DAILY_H22_RESULTS_DIR'] = str(RESULTS_DIR)
os.environ['MPLCONFIGDIR'] = '/content/matplotlib_cache'
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(EXP / 'requirements-colab.txt')], check=True)

def run_script(name, *args):
    command = [sys.executable, str(EXP / 'src' / name), *map(str, args)]
    print('RUN:', ' '.join(command))
    subprocess.run(command, check=True, env=os.environ.copy())

print('Experiment:', EXP)
print('Session data:', DATA_DIR)
print('Session results:', RESULTS_DIR)
try:
    import torch
    print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
          '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
except Exception as exc:
    print('Torch check failed:', exc)


## Importar os resultados do notebook 1

Selecione `daily_h22_transfer_to_neural.zip`, baixado pelo notebook 1. Isso
permite usar outra conta do Colab sem depender de uma pasta compartilhada.

In [ ]:
# Importar o ZIP gerado pelo notebook 1
import shutil
from pathlib import Path

transfer_name = 'daily_h22_transfer_to_neural.zip'
transfer_path = Path('/content') / transfer_name
if not transfer_path.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        if transfer_name not in uploaded:
            raise FileNotFoundError(
                f'Selecione exatamente {transfer_name}; recebido: {list(uploaded)}')
        transfer_path.write_bytes(uploaded[transfer_name])
    except ImportError as exc:
        raise FileNotFoundError(
            f'Fora do Colab, copie {transfer_name} para /content antes de continuar.') from exc

shutil.unpack_archive(str(transfer_path), str(PERSIST))
required = [
    DATA_DIR / 'daily_panel.parquet',
    DATA_DIR / 'split_manifest.csv',
    RESULTS_DIR / 'forecasts/rw.parquet',
    RESULTS_DIR / 'forecasts/har.parquet',
    RESULTS_DIR / 'forecasts/garch_11.parquet',
    RESULTS_DIR / 'forecasts/vix.parquet',
    RESULTS_DIR / 'forecasts/vix_calibrated.parquet',
    RESULTS_DIR / 'forecasts/xgboost.parquet',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('ZIP de transferência incompleto:\n' + '\n'.join(missing))
print('Transfer imported successfully:', transfer_path)


## Configuração computacional

O orçamento foi dividido entre Optuna e os ajustes rolling para manter a
execução total dentro de 70 minutos. O modo final usa 25 trials, e não os 40 do
desenho anterior, porque busca e treinamento agora rodam na mesma sessão.

In [ ]:
RUN_MODE = 'final'  # 'smoke', 'validation' ou 'final'
TARGET_TRIALS = {'smoke': 3, 'validation': 10, 'final': 25}[RUN_MODE]
MAX_STEPS = {'smoke': 30, 'validation': 200, 'final': 400}[RUN_MODE]
OPTUNA_BUDGET_MINUTES = {'smoke': 5, 'validation': 15, 'final': 35}[RUN_MODE]
NEURAL_BUDGET_MINUTES = {'smoke': 5, 'validation': 10, 'final': 20}[RUN_MODE]
MAX_WINDOWS = {'smoke': 1, 'validation': 2, 'final': None}[RUN_MODE]
LOCAL_DB = RESULTS_DIR / 'hp_search/tsmixerx_daily.db'
print(RUN_MODE, TARGET_TRIALS, MAX_STEPS,
      OPTUNA_BUDGET_MINUTES, NEURAL_BUDGET_MINUTES)


## Busca Optuna

O objetivo é o QLIKE médio dos 22 horizontes na validação. A busca seleciona
arquitetura, input size, loss e número de features usando apenas a primeira
janela 8/2. O pruning chama `trial.report` e `trial.should_prune`.

In [ ]:
optuna_args = [
    '--target-trials', TARGET_TRIALS,
    '--max-steps', MAX_STEPS,
    '--timeout-minutes', OPTUNA_BUDGET_MINUTES,
    '--storage', LOCAL_DB,
]
if RUN_MODE == 'smoke':
    optuna_args += ['--smoke']
run_script('12_tsmixerx_optuna.py', *optuna_args)


## Melhor configuração

In [ ]:
import json, pandas as pd
best_path = RESULTS_DIR / 'hp_search/best_config.json'
if not best_path.exists():
    raise FileNotFoundError('A busca não produziu best_config.json')
best = json.loads(best_path.read_text())
print(json.dumps(best, indent=2))
trial_files = sorted((RESULTS_DIR / 'hp_search').glob('trials_*.csv'))
trials = pd.read_csv(trial_files[-1])
display(trials.tail(25))
print(trials['state'].value_counts(dropna=False))
if RUN_MODE == 'final' and len(trials) < TARGET_TRIALS:
    raise RuntimeError('Busca incompleta dentro do orçamento estimado.')


## Ajustes rolling TSMixerX e LSTM

A configuração vencedora é congelada e os pesos são reajustados em cada uma das
16 janelas. Os modelos, forecasts, inputs, rankings e históricos são salvos no
armazenamento local da sessão.

In [ ]:
neural_args = [
    '--models', 'tsmixerx', 'lstm',
    '--max-steps', MAX_STEPS,
    '--time-budget-minutes', NEURAL_BUDGET_MINUTES,
]
if MAX_WINDOWS is not None:
    neural_args += ['--max-windows', MAX_WINDOWS]
run_script('13_neural_rolling.py', *neural_args)


## Verificação de cobertura

In [ ]:
coverage = []
for name in ['tsmixerx', 'lstm']:
    path = RESULTS_DIR / f'forecasts/{name}.parquet'
    if path.exists():
        frame = pd.read_parquet(path)
        coverage.append({'model': name, 'windows': frame.window_id.nunique(),
                         'rows': len(frame)})
    else:
        coverage.append({'model': name, 'windows': 0, 'rows': 0})
coverage_df = pd.DataFrame(coverage)
display(coverage_df)
NEURAL_COMPLETE = bool((coverage_df.windows == 16).all())
if RUN_MODE == 'final' and not NEURAL_COMPLETE:
    raise RuntimeError('Cobertura neural incompleta dentro do orçamento estimado.')


## Explicabilidade temporal

Integrated Gradients cobre quatro janelas e quatro horizontes. Shapley Value
Sampling é aplicado a uma amostra menor na janela final.

In [ ]:
if NEURAL_COMPLETE:
    run_script('14_explainability.py', '--window-ids', 0, 5, 10, 15,
               '--origins-per-window', 3, '--horizons', 1, 5, 10, 22,
               '--n-steps', 32)
    run_script('14_explainability.py', '--window-ids', 15,
               '--origins-per-window', 1, '--horizons', 1, 22,
               '--n-steps', 16, '--run-shapley', '--shapley-samples', 10)


## Avaliação final

São calculados QLIKE, MSE, MAE e R² OOS por horizonte, DM, GW com HAC de 21
lags e MCS via `arch.bootstrap.MCS` em `h=22` com block size 22.

In [ ]:
if NEURAL_COMPLETE:
    run_script('15_evaluate_daily.py', '--require-complete', '--mcs-reps', 1000)
    display(pd.read_csv(RESULTS_DIR / 'metrics/metrics_h22.csv').sort_values('MSE'))
    display(pd.read_csv(RESULTS_DIR / 'metrics/model_coverage.csv'))


## Manifesto e verificação de recarga

In [ ]:
run_script('16_package_results.py')
manifest = RESULTS_DIR / 'artifact_manifest.json'
if manifest.exists():
    print(manifest.read_text()[:4000])

artifacts = sorted((RESULTS_DIR / 'models/tsmixerx').glob('*.pt'))
print('TSMixerX artifacts:', len(artifacts))
if artifacts:
    sys.path.insert(0, str(EXP / 'src'))
    from models import DirectVolatilityRegressor
    loaded, payload = DirectVolatilityRegressor.load(artifacts[-1])
    print('Reloaded:', artifacts[-1].name, '| input_size:', loaded.input_size,
          '| features:', len(payload['features']))


## Download dos resultados finais

O ZIP contém o painel, forecasts de todos os modelos, modelos neurais,
históricos, explicabilidade, métricas, figuras e manifesto. Os modelos
estatísticos completos permanecem no ZIP separado baixado pelo notebook 1.

In [ ]:
ARCHIVE_NAME = 'daily_h22_final_results'
# Empacotamento e download seguro do estado atual
import shutil
from pathlib import Path

archive_base = Path('/content') / ARCHIVE_NAME
output_file = shutil.make_archive(str(archive_base), 'zip', root_dir=PERSIST)
print('Archive created:', output_file)
try:
    from google.colab import files
    files.download(output_file)
    print("Downloaded:", output_file)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
